mode BATCH

In [0]:
from pyspark.sql import functions as F

# ------------------------------------------------------------
# Configuration UC + Volume Bronze
# ------------------------------------------------------------

spark.sql("USE CATALOG main")
spark.sql("USE SCHEMA bronze")

bronze_volume_path = "/Volumes/main/bronze/bronze_volume"
kaggle = f"{bronze_volume_path}/kaggle"
landing_zone = f"{bronze_volume_path}/landing_zone"

print(f"Bronze Volume : {bronze_volume_path}")
print(f"Kaggle Batch : {kaggle}")
print(f"Landing Zone  : {landing_zone}")

# ------------------------------------------------------------
# Charger le dataset Kaggle (CSV)
# ------------------------------------------------------------
# /Volumes/main/bronze/bronze_volume/kaggle/creditcard.csv

kaggle_path = f"{bronze_volume_path}/kaggle/creditcard.csv"

df_bronze = (
    spark.read
         .option("header", True)
         .option("inferSchema", True)
         .csv(kaggle_path)
)

print("✓ Dataset Kaggle chargé")
df_bronze.show(5)

df_bronze.printSchema()

# ------------------------------------------------------------
# 4. Écriture dans la table Bronze UC
# ------------------------------------------------------------

df_bronze.write.format("delta").mode("overwrite").saveAsTable("transactions_bronze")

print("✓ Table Bronze créée : main.bronze.transactions_bronze")

# ------------------------------------------------------------
# 5. Export batch dans la landing_zone (optionnel)
# ------------------------------------------------------------
# Cela permet au pipeline streaming de lire aussi le batch initial.

df_bronze.write.mode("overwrite").option("header", True).csv(landing_zone)

print("✓ Export batch vers landing_zone pour le streaming")
